# N09 — CNN and Offline Transfer-Learning Readiness

**Sessions:** 49–50  
**Objective:** Train a tiny CNN, inspect visual errors, and validate an offline pretrained-asset contract without network downloads.

## Task contract
Run top-to-bottom from a fresh kernel. Do not install packages inside the notebook. Record one error or misconception and complete the independent transfer before submission.


In [ ]:
from __future__ import annotations
import hashlib, json, platform, random
from pathlib import Path
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", platform.python_version())
print("Working directory:", Path.cwd())


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
import matplotlib.pyplot as plt

torch.set_num_threads(1); torch.manual_seed(SEED); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng=np.random.default_rng(SEED)
images=[]; labels=[]
for i in range(120):
    canvas=np.zeros((24,24),dtype=np.float32); label=i%2
    if label==0: canvas[7:17,7:17]=1
    else:
        yy,xx=np.ogrid[:24,:24]; canvas[(xx-12)**2+(yy-12)**2<=30]=1
    canvas=np.clip(canvas+rng.normal(0,.12,canvas.shape),0,1)
    images.append(canvas); labels.append(label)
X=torch.tensor(np.array(images)[:,None],dtype=torch.float32); y=torch.tensor(labels)
perm=torch.randperm(len(X),generator=torch.Generator().manual_seed(SEED)); tr=perm[:90]; va=perm[90:]
train=DataLoader(TensorDataset(X[tr],y[tr]),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
model=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(8*12*12,2)).to(device)
opt=torch.optim.Adam(model.parameters(),lr=.01); loss_fn=nn.CrossEntropyLoss()
for epoch in range(8):
    model.train()
    for xb,yb in train:
        xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
model.eval();
with torch.no_grad(): pred=model(X[va].to(device)).argmax(1).cpu(); acc=(pred==y[va]).float().mean().item()
print('validation accuracy:',acc); assert acc>.85


In [ ]:
wrong=va[pred!=y[va]]
print('wrong examples:',len(wrong))
fig,axes=plt.subplots(1,min(5,max(1,len(wrong))),figsize=(10,2))
axes=np.atleast_1d(axes)
show=wrong[:len(axes)] if len(wrong) else va[:1]
for ax,idx in zip(axes,show): ax.imshow(X[idx,0],cmap='gray'); ax.set_title(f'true={int(y[idx])}'); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
asset_contract = {
    'checkpoint_required': True,
    'network_downloads_allowed': False,
    'required_metadata': ['filename','sha256','source','licence','preprocessing','frozen_layers'],
}
print(asset_contract)
assert asset_contract['network_downloads_allowed'] is False


## Independent transfer
Use a teacher-provided, locally cached, rule-permitted pretrained checkpoint. Record its filename, SHA-256 hash, source/licence, preprocessing, frozen/unfrozen layers, and compare it with the tiny-CNN baseline under the same split.


## Fresh-run checklist

- [ ] Restart kernel and run all cells in order.
- [ ] Confirm assertions pass.
- [ ] Record package versions and seed.
- [ ] Save the required artifact with a relative path.
- [ ] Add an error-log entry and AI-use note.
- [ ] Explain one teacher-selected cell without reading it.
